## Programming part of Homework 4 (Data Structures, Fall 2025)

## Name:陳彥岑
## Student ID Number:113820031

### Programming problem 1
**Univariate polynomial** of degree $d$ has the form $$c_dx^d+c_{d-1}x^{d-1}+\cdots + c_2x^2+c_1x+c_0,$$ where $c_d\not= 0$. The $c_i$'s are the \emph{coefficients}, and $d, d-1, \cdots$ are the \emph{exponents}. By definition, $d$ is a nonnegative integer. In this exercise, we assume that all $c_i$s are integers. We represent each polynomial as a *linear list* of coefficients and would like to have some operations (functions) on the polynomials. The first node in the list represents the first terms in the polynomial, the second node represents the second terms, and so forth.

Each node contains three fields: *the term's coefficient*, *the term's power*, and *a pointer to the next term*. Write a Python program, that first reads the input file, `inFile.txt`, which has three lines and then performs the indicated operation. The first line is an integer representing the operation defined as below. The second line is the first polynomial and the next line is the second polynomial. The input polynomial, say $4x^3-2x+1$, will be represented as `4x^3-2x+1`. The functions include the following operations:
1. `add`: Add two input polynomials.
2. `subtract`: Subtract the second polynomial from the first one.
3. `multiply`: Multiply two polynomials.
4. `divide`: Divide the first polynomial by the second one and return the quotient.
The input file thus may be

```
2
4x^3-2x+1
3x^2+x+4
```

Your output will be `4x^3-3x^2-3x-3`. Please see the running example in the end of this template.

Python has a built-in package called **re**, which can be used to work with Regular Expressions and provides regular expression matching operations similar to those found in Perl. One can use this package for parsing the input strings. For more details, please refer to
[PYTHON Regular Expression](https://www.w3schools.com/python/python_regex.asp) and 
[Python RegEx](https://docs.python.org/3/library/re.html).

First, we build up the linked list structure for representing polynomials. Two classes will be defined: `Node` and `Poly_List`.

In [10]:
# Build up the linked list structure for representing polynomials
import re # When you have imported the "re" module, you can start using regular expressions

# Define the class of node in the linked list used for polynomial representation
# Node class 
class Node:
    def __init__(self,c,exp):
        self.coefficient = float(c) # 係數
        self.exponential = int(exp) # 指數
        self.next = None
        
    # get the coefficient in this node
    def getCoefficient(self):
        return self.coefficient
    
    # get the exponent in this node
    def getExponential(self):
        return self.exponential
    
    # get the next node
    def getNext(self):
        return self.next
    
    # set the coefficient and exponent to this node
    def setData(self,c,exp):
        self.coefficient = float(c)
        self.exponential = int(exp)

    # set the coefficient to this node only
    def setCoefficient(self,c):
        self.coefficient = float(c)

    # set the exponent to this node only
    def setExponential(self,exp):
        self.exponential = int(exp)

    # assign the next node to this node 
    def setNext(self,newnext):
        self.next = newnext

# Define the class of the linked list used for polynomial representation
# List class 
class Poly_List:
    def __init__(self):
        self.head = None
        self.tail = None

    # methods for managing the list
    def isEmpty(self):
        return self.head == None

    def size(self):
        current = self.head
        count = 0
        while current != None:
            count += 1
            current = current.getNext()
        return count

    def isHead(self, node):
        return node == self.head

    def isTail(self, node):
        return node == self.tail
    
    # get the head of the list
    def getHead(self):
        return self.head

    # get the tail of the list
    def getTail(self):
        return self.tail

    # set the head of the list
    def setHead(self, node):
        self.head = node
        
    # set the tail of the list
    def setTail(self, node):
        self.tail = node

    # get the degree of the polynomial
    def polyDegree(self):
        if self.isEmpty():
            return -1
        return self.head.getExponential()

    # insert a term (node) after node p
    def insertAfter(self,p,c,exp):
        new_node = Node(c, exp)
        new_node.setNext(p.getNext())
        p.setNext(new_node)
        if p == self.tail:
            self.tail = new_node

    # insert a term (node) at head
    def insertAtHead(self,c,exp):
        new_node = Node(c, exp)
        new_node.setNext(self.head)
        self.head = new_node
        if self.tail is None:
            self.tail = new_node

    # insert a term (node) at tail
    def insertAtTail(self,c,exp):
        new_node = Node(c, exp)
        if self.isEmpty():
            self.head = new_node
            self.tail = new_node
        else:
            self.tail.setNext(new_node)
            self.tail = new_node

    # delete a term (node) at head
    def deleteAtHead(self):
        if self.isEmpty():
            return
        
        removed_node = self.head
        self.head = self.head.getNext()
        if self.head is None:
            self.tail = None
        
        return removed_node

    # Method for adding the missing terms and may be used for division
    def paddingPoly(self):
        if self.isEmpty():
            return
            
        current = self.head
        
        while current.getNext() is not None:
            # 檢查當前節點和下一個節點之間的次數差是否大於 1
            if current.getExponential() - current.getNext().getExponential() > 1:
                missing_exp = current.getExponential() - 1
                # 插入一個係數為 0 的項
                self.insertAfter(current, 0, missing_exp)
            current = current.getNext()

    # This method is used for multiplying the polynomial by a constant m or
    # lifting all terms by a degree d
    def timeConst_liftDegree(self, m, d):
        new_poly = self.copy() # 複製一份新的多項式進行操作
        
        current = new_poly.head
        while current is not None:
            # 乘以常數 m
            current.setCoefficient(current.getCoefficient() * m)
            # 提升次數 d
            current.setExponential(current.getExponential() + d)
            current = current.getNext()
            
        new_poly.verifyDegree() # 處理係數為 0 的情況
        return new_poly

    # Method to verify the degree of polynomial for getting rid of the higher
    # terms with 0 as coefficients

    def verifyDegree(self):
        
        # 1. 刪除頭部係數為 0 的項 (最高次項)
        while self.head is not None and abs(self.head.getCoefficient()) < 1e-9: # 浮點數比較使用容忍度
            self.deleteAtHead()
            
        if self.isEmpty():
            # 避免空多項式
            return
            
        # 2. 刪除列表中間係數為 0 的項
        current = self.head
        while current.getNext() is not None:
            if abs(current.getNext().getCoefficient()) < 1e-9:
                # 刪除下一個節點
                current.setNext(current.getNext().getNext())
                if current.getNext() is None:
                    self.tail = current # 更新 tail
                # 不移動 current，因為 current.next 已經換成新的節點，需要再次檢查
            else:
                current = current.getNext()
                
    # This method returns a copy of the polynomail with a new list
    def copy(self):
        new_list = Poly_List()
        current = self.head
        while current is not None:
            # 複製係數和指數，並在尾部插入
            new_list.insertAtTail(current.getCoefficient(), current.getExponential())
            current = current.getNext()
        return new_list
    
    # This is used to print the list for represented polynomial
    def printPoly_List(self):
        current = self.head
        parts = []
        while current is not None:
            parts.append(f"({current.getCoefficient()},{current.getExponential()})")
            current = current.getNext()
        print(" -> ".join(parts))

    # This prints the polynomial in a given format
    def printPolynomial(self):
        self.verifyDegree() # 輸出前清理 0 係數項
        
        if self.isEmpty():
            print("0.0")
            return
            
        current = self.head
        output = ""
        is_first_term = True
        
        while current is not None:
            coeff = current.getCoefficient()
            exp = current.getExponential()
            
            # 跳過係數接近 0 的項
            if abs(coeff) < 1e-9:
                current = current.getNext()
                continue
            
            # 處理符號
            if not is_first_term:
                if coeff > 0:
                    output += "+"
                elif coeff < 0:
                    output += "" # 負號會在係數中包含
            
            # 格式化係數
            abs_coeff = abs(coeff)
            
            # 處理係數部分
            if exp == 0 or abs_coeff != 1.0:
                coeff_str = f"{coeff:.1f}"
                if coeff_str.endswith(".0"): # 如果是整數，可以簡化顯示
                    coeff_str = str(int(coeff))
                elif len(coeff_str.split('.')[-1]) > 1:
                    coeff_str = f"{coeff:.3f}"
                    
                # 處理第一個非零係數項的符號
                if is_first_term and coeff < 0:
                     output += coeff_str
                elif not is_first_term and coeff < 0:
                     output += coeff_str
                else:
                    output += coeff_str
            
            elif abs_coeff == 1.0:
                if is_first_term and coeff == -1.0:
                    output += "-"
                elif coeff == -1.0:
                    output += "-" # 因為前面已經加了 - 符號，但這裏是 `output += ""`
                elif coeff == 1.0:
                    pass # 係數為 1 時不顯示
                
            
            # 處理 x 和指數部分
            if exp == 1:
                output += "x"
            elif exp > 1:
                output += f"x^{exp}"

            is_first_term = False
            current = current.getNext()
        
        # 處理結果為空的情況（例如：所有項都是 0）
        if output == "":
             print("0.0", end="")
        else:
            print(output, end="")
        print() # 


Then, we may provide the functions for helping read and parse the input file to have the input operation and polynnomials. 
The `read_lines()` function reads the lines into and returns a list of strings. 
Function `read_string(s)` parses an input string to a polynomial with ***linked list representation***. 

In [6]:
# functions for reading and parsing the input file to have the input polynnomials and operation 
# function for reading lines in the input text file into a list of strings      
def read_lines():
    try:
        # 假設輸入檔案名為 inFile.txt
        with open('inFile.txt', 'r') as f:
            lines = [line.strip() for line in f.readlines()]
        return lines
    except FileNotFoundError:
        print("Error: inFile.txt not found.")
        return []
    
# function for parsing the line into polynomial with linked list representation 
def read_string(s):
    # 正則表達式，用於匹配形如 (+/-)C(x^E) 的項。
    # (\+|\-)? : 匹配可選的 + 或 - 符號
    # ((\d*\.?\d*)|(\d*)) : 匹配係數，可以是整數或浮點數
    # (x(\^\d+)?)? : 匹配 x 和可選的指數部分
    
    # 修正後的正則表達式，能更好地處理各種情況，例如 x, -x, x^3, -2.5x^2, 5
    # r"([+-]?[ ]?([0-9]*\.?[0-9]*)x(\^[0-9]+)?)|([+-]?[ ]?[0-9]*\.?[0-9]+)"
    
    # 嘗試使用更穩健的表達式來分割
    # 確保在操作符號 (+/-) 後分割字串
    # 這個表達式會在每個加號或減號前分割，但不會分割第一個項（除非它前面有符號）
    terms = re.findall(r"([+-]?[ ]?\b\d*[\.]?\d*x?\^?\d*)", s.replace(' ', ''))
    
    poly_list = Poly_List()
    
    # 用於儲存 (指數, 係數) 的字典，方便排序和合併同類項
    terms_dict = {}

    for term_str in terms:
        term_str = term_str.strip()
        if not term_str:
            continue
            
        coeff = 1.0
        exp = 0
        
        # 1. 解析指數
        if 'x' in term_str:
            exp_match = re.search(r'x\^(\d+)', term_str)
            if exp_match:
                exp = int(exp_match.group(1))
            else: # x^1
                exp = 1
        
        # 2. 解析係數
        
        # 移除 x^exp 部分，只保留係數前綴
        coeff_part = re.sub(r'x(\^\d+)?', '', term_str).strip()
        
        if coeff_part == '-' or coeff_part == '': # 處理 -x 或 x
             coeff = -1.0 if coeff_part == '-' else 1.0
        else:
            try:
                # 嘗試將係數部分轉換為浮點數
                coeff = float(coeff_part)
            except ValueError:
                # 如果係數部分為空 (例如 x^3)，則係數為 1.0 (或 -1.0)
                if term_str.startswith('-'):
                    coeff = -1.0
                elif term_str.startswith('+'):
                    coeff = 1.0
                else:
                    coeff = 1.0

        if abs(coeff) < 1e-9: # 係數為 0 的項忽略
            continue
            
        # 將項添加到字典中，如果指數已存在則合併係數
        terms_dict[exp] = terms_dict.get(exp, 0.0) + coeff
    
    # 3. 根據指數從大到小排序並構建鏈結列表
    sorted_terms = sorted(terms_dict.items(), key=lambda item: item[0], reverse=True)
    
    for exp, coeff in sorted_terms:
        if abs(coeff) >= 1e-9:
            poly_list.insertAtTail(coeff, exp)

    return poly_list

Below, the functions for polynomial operations with two input polynomials are provided:
1. `add()`: Add two input polynomials.
2. `subtract()`: Subtract the second polynomial from the first one.
3. `multiply()`: Multiply two polynomials.
4. `divide()`: Divide the first polynomial by the second one and return the quotient and remainder.

**Note that** since `divide()` returns two resulting polynomails. We therefore have all the functions for operations return two polynomials. If there is only one resulting polynomial, we use `None` object for the second polynomail to return.

In [7]:
# functions for polynomial operations
def merge_polys(poly1, poly2, sub=False):
    result_poly = Poly_List()
    p1 = poly1.head
    p2 = poly2.head
    
    while p1 is not None or p2 is not None:
        c = 0.0
        exp = 0
        
        if p1 is None: # poly1 已遍歷完，只剩 poly2
            c = -p2.getCoefficient() if sub else p2.getCoefficient()
            exp = p2.getExponential()
            p2 = p2.getNext()
            
        elif p2 is None: # poly2 已遍歷完，只剩 poly1
            c = p1.getCoefficient()
            exp = p1.getExponential()
            p1 = p1.getNext()
            
        else:
            exp1 = p1.getExponential()
            exp2 = p2.getExponential()
            
            if exp1 == exp2: # 次數相同，合併係數
                c = p1.getCoefficient() + (-p2.getCoefficient() if sub else p2.getCoefficient())
                exp = exp1
                p1 = p1.getNext()
                p2 = p2.getNext()
                
            elif exp1 > exp2: # poly1 次數較高
                c = p1.getCoefficient()
                exp = exp1
                p1 = p1.getNext()
                
            else: # poly2 次數較高
                c = -p2.getCoefficient() if sub else p2.getCoefficient()
                exp = exp2
                p2 = p2.getNext()

        if abs(c) >= 1e-9: # 係數不為 0 才新增
            result_poly.insertAtTail(c, exp)
            
    # 確保最終多項式的 head 和 tail 正確設置（如果列表為空）
    if result_poly.isEmpty():
        result_poly.insertAtTail(0, 0) # 結果為 0
    else:
        result_poly.verifyDegree()

    return result_poly


# adding two polynomials
# adding two polynomials
def add(poly1,poly2):
    result = merge_polys(poly1, poly2, sub=False)
    return result, None

# substracting poly2 from poly1 
def substract(poly1,poly2):
    result = merge_polys(poly1, poly2, sub=True)
    return result, None

# multiplying two polynomials
def multiply(poly1,poly2):
    result_poly = Poly_List()
    # 用於儲存 (指數, 係數) 的字典
    terms_dict = {}

    p1 = poly1.head
    while p1 is not None:
        p2 = poly2.head
        while p2 is not None:
            # 乘法規則: (c1 * x^e1) * (c2 * x^e2) = (c1*c2) * x^(e1+e2)
            new_coeff = p1.getCoefficient() * p2.getCoefficient()
            new_exp = p1.getExponential() + p2.getExponential()
            
            # 合併同類項
            terms_dict[new_exp] = terms_dict.get(new_exp, 0.0) + new_coeff
            
            p2 = p2.getNext()
        p1 = p1.getNext()

    # 根據指數從大到小排序並構建鏈結列表
    sorted_terms = sorted(terms_dict.items(), key=lambda item: item[0], reverse=True)
    
    for exp, coeff in sorted_terms:
        if abs(coeff) >= 1e-9:
            result_poly.insertAtTail(coeff, exp)
            
    if result_poly.isEmpty():
        result_poly.insertAtTail(0, 0)
    else:
        result_poly.verifyDegree()
        
    return result_poly, None

# dividng poly1 by poly2 and then returning the quotient and remainder
# 實作長除法 (Long Division)
def divide(poly1,poly2):
    # 多項式 B 不能為 0
    poly2.verifyDegree()
    if poly2.isEmpty() or abs(poly2.head.getCoefficient()) < 1e-9:
        raise ZeroDivisionError("Division by zero polynomial")

    # A = 被除數 (Dividend), B = 除數 (Divisor)
    A = poly1.copy()
    B = poly2.copy()
    Q = Poly_List() # 商 (Quotient)
    R = A.copy() # 餘數 (Remainder)

    # 確保 R 始終有最高的次數
    R.verifyDegree() 
    
    # 如果 Degree(A) < Degree(B)，則 Q=0, R=A
    if R.polyDegree() < B.polyDegree():
        Q.insertAtHead(0, 0) # 商為 0
        return Q, R

    # 長除法循環
    while R.polyDegree() >= B.polyDegree():
        
        # 1. 計算最高次項相除的結果 (C_k * x^k)
        # 獲取 R 和 B 的首項係數和指數
        r_coeff = R.head.getCoefficient()
        r_exp = R.head.getExponential()
        b_coeff = B.head.getCoefficient()
        b_exp = B.head.getExponential()

        # C_k = (r_coeff / b_coeff), k = (r_exp - b_exp)
        # C_k * x^k 是商 Q 的新一項
        q_coeff = r_coeff / b_coeff
        q_exp = r_exp - b_exp
        
        # 2. 將新項加入商 Q
        # 為了保持 Q 始終按次數遞減，我們需要一個輔助列表
        new_term_poly = Poly_List()
        new_term_poly.insertAtHead(q_coeff, q_exp)

        # 將新項添加到 Q 的尾部（這樣保持了次數遞減）
        Q.insertAtTail(q_coeff, q_exp) 
        
        # 3. M = 新項 * B
        # M = (q_coeff * x^q_exp) * B
        M = B.timeConst_liftDegree(q_coeff, q_exp)
        
        # 4. R = R - M
        # 由於 merge_polys 實作的是加法，我們用 R + (-M) 來實現 R - M
        M_neg = M.timeConst_liftDegree(-1.0, 0) # M 乘以 -1
        R = merge_polys(R, M_neg, sub=False) # R + (-M)
        
        R.verifyDegree() # 清理 R 中係數為 0 的項

    # Q 和 R 的清理
    if Q.isEmpty():
        Q.insertAtHead(0, 0)
    
    if R.isEmpty():
        R.insertAtHead(0, 0)
    
    Q.verifyDegree()
    R.verifyDegree()
    
    return Q, R

Last, we perform the operations according to the input file, `inFile.txt`. Function `poly_operation()` can be as the main program entry and first derive the operation with the derived list of strings from function `read_lines()`. Then, `operation_selection()` is called to perform the corresponding operation. **Note that** there will be two polynomials returned for each operation. Last, it prints out the result. The whole program will be executed by calling `poly_operation()` with the input file, `inFile.txt`. One can change the content in the input file for different cases. 

***Basically, the following cell can be kept without change if you would like to follow it for programming. Of course, you can have your own code for this part. However, the function name of the main program entry, `poly_operation()` can not be changed.***

In [13]:
# main program area
# function to call the corresponding operation and return two polynomial
def operation_selection(operation, poly1, poly2):
    switcher = {
        1: add,
        2: substract,
        3: multiply,
        4: divide,
    }
    # Get the function from switcher dictionary
    func = switcher.get(operation, lambda: (Poly_List(), Poly_List())) # 錯誤時返回兩個空多項式

    # Execute the function
    return func(poly1,poly2)

# program entry
# function for starting the task
def poly_operation():
    #
    # read the input information from the default input text file
    #
    strings=read_lines()

    if len(strings) < 3:
        print("Error: Input file should contain at least 3 lines.")
        return

    #
    # obtain the operation: 1. add; 2. substract; 3. Multiply; 4. Divide
    # and print it out
    #
    try:
        operation=int(strings[0])
    except ValueError:
        print("Error: First line must be an integer operation code.")
        return
        
    operations={
        1: 'add',
        2: 'substract',
        3: 'multiply',
        4: 'divide'
    }
    
    poly1_str = strings[1]
    poly2_str = strings[2]
    
    print(poly1_str, operations.get(operation), poly2_str)

    #
    # parse strings 1 and 2 to derive the input polynomials and represent them with
    # linked lists
    #
    poly1=read_string(poly1_str)
    poly2=read_string(poly2_str)
    
    #
    # perform the operation and two polynomials are returned.
    #
    try:
        r1, r2=operation_selection(operation, poly1, poly2)
    except ZeroDivisionError as e:
        print(f"Error: {e}")
        return
    
    #
    # print out the result
    #
    if (operation==4):
        print("The quotient is:", end="")
        r1.printPolynomial()
        print("The remainder is:", end="")
        r2.printPolynomial()
    else:
        print("The result is:", end="")
        r1.printPolynomial()        

# execute the program with the input file inFile.txt
poly_operation()

4x^3-2x+1 add -3x^2+x+4
The result is:4x^3-3x^2-x+5
